<a href="https://colab.research.google.com/github/linda-bsharat/telco-customer-churn-prediction/blob/main/notebooks/01_data_cleaning_and_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Telco Customer Churn Project**







###**01- Data cleaning and handling missing values**
The dataset contains approximately 70,000 customer records and 21 features describing customer demographics, service subscriptions, account information, and billing details.

Each row represents one customer.

The target variable is:



*   Churn (Yes/No) — indicating whether the customer left the service.

Key features include:

*   Demographics (gender, senior citizen, partner, dependents)
*   Account information (tenure, contract type, payment method)
*   Services (internet service, tech support, streaming services)
*   Billing information (monthly charges, total charges)

###  1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

###  2. Load Data

In [ ]:
# Load the dataset

url = "https://raw.githubusercontent.com/linda-bsharat/telco-customer-churn-prediction/main/data/telco_customer_data_v2.csv"
df = pd.read_csv(url)

# Display first rows

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,CUST00001,Male,0,No,Yes,3.0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,68.61,205.83,Yes
1,CUST00002,Male,1,Yes,No,2.0,Yes,Yes,DSL,No,...,No internet service,Yes,NaN,No,One year,Yes,Bank transfer (automatic),23.15,46.3,No
2,CUST00003,Female,No,No,No,42.0,Yes,Yes,DSL,No,...,No,NaN,Yes,Yes,Month-to-month,No,Electronic check,42.63,1790.46,Yes
3,CUST00004,Female,0,No,Yes,40.0,Yes,Yes,Fiber optic,No,...,Yes,No,No,No internet service,Month-to-month,No,Electronic check,75.04,3001.6,No
4,CUST00005,Male,Yes,Yes,Yes,17.0,Yes,NaN,Fiber optic,Yes,...,Yes,No,No internet service,No,Two year,Yes,Electronic check,22.38,380.46,Yes


###   3. Initial Data Exploration

In [ ]:
# Check dataset shape
df.shape

(70000, 21)

In [ ]:
# View column names
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [ ]:
# Check data types and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        70000 non-null  object 
 1   gender            69252 non-null  object 
 2   SeniorCitizen     69341 non-null  object 
 3   Partner           66470 non-null  object 
 4   Dependents        66435 non-null  object 
 5   tenure            69433 non-null  float64
 6   PhoneService      70000 non-null  object 
 7   MultipleLines     68132 non-null  object 
 8   InternetService   70000 non-null  object 
 9   OnlineSecurity    67078 non-null  object 
 10  OnlineBackup      67253 non-null  object 
 11  DeviceProtection  67106 non-null  object 
 12  TechSupport       67267 non-null  object 
 13  StreamingTV       67173 non-null  object 
 14  StreamingMovies   67215 non-null  object 
 15  Contract          70000 non-null  object 
 16  PaperlessBilling  70000 non-null  object

In [ ]:
# Statistical summary
df.describe()

,tenure,MonthlyCharges
count,69433.000000,69612.000000
mean,30.516858,60.588548
std,89.873767,111.509588
min,-10.000000,18.000000
25%,10.000000,29.670000
50%,20.000000,41.190000
75%,35.000000,63.882500
max,999.000000,1499.770000


###  3. Data Cleaning (Handle Missing Values)


In [ ]:
df.isna().sum().sort_values(ascending=False).head(15)

,0
PaymentMethod,3569
Dependents,3565
Partner,3530
OnlineSecurity,2922
DeviceProtection,2894
StreamingTV,2827
StreamingMovies,2785
OnlineBackup,2747
TechSupport,2733
MultipleLines,1868



*   ###  Cleaning Binary Columns (Yes/No)



In [ ]:
# List of service-related columns that contain Yes/No values

service_cols = [
    "Dependents",
    "Partner",
    "OnlineSecurity",
    "DeviceProtection",
    "StreamingTV",
    "StreamingMovies",
    "OnlineBackup",
    "TechSupport",
    "MultipleLines",
]

# Fill missing values with "No"

df[service_cols] = df[service_cols].fillna("No")





*   ### Missing data




In [ ]:
# Check the columns with the highest number of missing values

df.isna().sum().sort_values(ascending=False).head(15)

,0
PaymentMethod,3569
TotalCharges,1062
gender,748
SeniorCitizen,659
tenure,567
MonthlyCharges,388
Dependents,0
customerID,0
Partner,0
OnlineSecurity,0


*   ### Fix TotalCharges type

In [ ]:
# Convert TotalCharges column to numeric
# Any invalid values will be converted to NaN

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

*  ### Handle numeric missing (tenure, MonthlyCharges)

In [ ]:
# Fill missing values in numerical columns using the median

df["tenure"] = df["tenure"].fillna(df["tenure"].median())
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(df["MonthlyCharges"].median())

*   ### Impute TotalCharges using tenure * MonthlyCharges

In [ ]:
# Create a mask to locate rows where TotalCharges is missing

mask = df["TotalCharges"].isna()

# Estimate missing TotalCharges using: TotalCharges ≈ tenure * MonthlyCharges

df.loc[mask, "TotalCharges"] = (
    df.loc[mask, "tenure"] * df.loc[mask, "MonthlyCharges"]
)

In [ ]:
# Check remaining missing values after handling TotalCharges

df.isna().sum().sort_values(ascending=False)

,0
PaymentMethod,3569
gender,748
SeniorCitizen,659
Partner,0
customerID,0
Dependents,0
tenure,0
MultipleLines,0
PhoneService,0
OnlineSecurity,0


*   ### Handle demographic missing values




In [ ]:
# Fill missing values in categorical columns using the most frequent value (mode)

df["gender"] = df["gender"].fillna(df["gender"].mode()[0])
df["SeniorCitizen"] = df["SeniorCitizen"].fillna(df["SeniorCitizen"].mode()[0])

In [ ]:
# Check again for remaining missing values

df.isna().sum().sort_values(ascending=False).head(3)

,0
PaymentMethod,3569
gender,0
SeniorCitizen,0


In [ ]:
# Inspect the distribution of values in PaymentMethod column

df["PaymentMethod"].value_counts()

,count
PaymentMethod,
Electronic check,27841
Bank transfer (automatic),14102
Mailed check,13963
Credit card (automatic),10400
BANK TRANSFER,125


In [ ]:
# Display rows where PaymentMethod values are missing
df[df["PaymentMethod"].isna()]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
88,CUST00089,Female,1,No,No,14.0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,NaN,37.44,524.16,Yes
90,CUST00091,Male,0,No,No,22.0,Yes,No,DSL,No,...,No,No,No internet service,No,Month-to-month,Yes,NaN,41.19,906.18,Yes
109,CUST00110,FEMALE,No,No,No,11.0,Yes,Yes,DSL,No,...,No internet service,No,No,No internet service,Two year,Yes,NaN,18.39,202.29,No
113,CUST00114,Female,0,Yes,Yes,21.0,No,No phone service,Fiber optic,No,...,No,No,No,No,Two year,No,NaN,25.91,544.11,No
159,CUST00160,Male,0,No,Yes,10.0,Yes,No,Fiber optic,No,...,Yes,No,No,No,Month-to-month,Yes,NaN,29.91,299.10,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69874,CUST69875,Male,0,No,Yes,18.0,Yes,Yes,Fiber optic,No,...,Yes,No,No,Yes,One year,Yes,NaN,41.19,741.42,No
69901,CUST69902,Female,0,No,Yes,30.0,Yes,No,DSL,No,...,No,No,No,No,Two year,No,NaN,46.09,1382.70,No
69933,CUST69934,Female,No,Yes,Yes,35.0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,NaN,41.19,1441.65,No
69936,CUST69937,Male,0,No,No,7.0,Yes,No,DSL,No,...,Yes,No,No,No,One year,Yes,NaN,29.43,206.01,Yes



    
*   ### Handle PaymentMethod missing values


In [ ]:
# Fill missing values in PaymentMethod using the most frequent category

df["PaymentMethod"] = df["PaymentMethod"].fillna(df["PaymentMethod"].mode()[0])

In [ ]:
# Verify that no missing values remain in the dataset

df.isna().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0




*   ### FIXING GENDER COLUMS




In [ ]:
# Standardize gender values by removing spaces and formatting text
df['gender'] = df['gender'].str.strip().str.capitalize()
# Replace shorthand gender values with standard labels
df['gender'] = df['gender'].replace({'F': 'Female', 'M': 'Male'})
# Correct inconsistent gender label
df['gender'] = df['gender'].replace({ 'Man': 'Male'})

In [ ]:
# Check the distribution of gender values after cleaning
df["gender"].value_counts()

,count
gender,
Female,35535
Male,34465




*   ### FIXING SENIOR CITIZEN COLUMNS




In [ ]:
# Convert SeniorCitizen column to binary numeric values

df["SeniorCitizen"] = df["SeniorCitizen"].replace({
    "Yes": 1,
    "No": 0,
    "not senior": 0})
# Ensure the column is stored as integer type
df["SeniorCitizen"] = df["SeniorCitizen"].astype(int)

In [ ]:
# Verify the distribution of SeniorCitizen values

df["SeniorCitizen"].value_counts()

,count
SeniorCitizen,
0,56053
1,13947




*   ###  Target Variable Cleaning (Churn)




In [ ]:
# Check the distribution of the target variable (Churn)
df["Churn"].value_counts()

,count
Churn,
No,37057
Yes,32713
Unknown,54
CHURNED,46
N,46
NO CHURN,45
Y,39




*   ### CONVARTE DATA IN CHURN



In [ ]:
# Enable strict behavior for future pandas type conversions

pd.set_option('future.no_silent_downcasting', True)

# Standardize different churn labels into binary values

df['Churn'] = df['Churn'].replace({
    'Yes': 1, 'Y': 1, 'CHURNED': 1,
    'No': 0, 'N': 0, 'NO CHURN': 0,
    'Unknown': None
})

# Remove rows where churn status is unknown
df = df.dropna(subset=['Churn'])
# Convert churn column to integer type
df['Churn'] = df['Churn'].astype(int)



In [ ]:
# Check the final distribution of the churn target variable
df["Churn"].value_counts()


,count
Churn,
0,37148
1,32798




*   ### Standardize PaymentMethod




In [ ]:
# Replace inconsistent naming of payment method with a standardized format

df['PaymentMethod'] = df['PaymentMethod'].replace({
    'BANK TRANSFER': 'Bank transfer (automatic)'
})

# Display the count of each payment method to verify the cleaning step

df['PaymentMethod'].value_counts()

,count
PaymentMethod,
Electronic check,31379
Bank transfer (automatic),14215
Mailed check,13959
Credit card (automatic),10393


*   ### Check unique values in binary columns



In [ ]:
# List of binary columns that contain Yes/No values
yes_no_columns = [
    'Partner',
    'Dependents',
    'PhoneService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'PaperlessBilling'
]

# Loop through each column and print its unique values
# This helps detect inconsistencies such as extra spaces, different spellings, or unexpected values

for col in yes_no_columns:
    print(f"{col} unique values:", df[col].unique())

Partner unique values: ['No' 'Yes']
Dependents unique values: ['Yes' 'No']
PhoneService unique values: ['Yes' 'No']
OnlineSecurity unique values: ['No internet service' 'No' 'Yes' 'Y' 'True']
OnlineBackup unique values: ['No internet service' 'No' 'Yes' 'Y' 'True']
DeviceProtection unique values: ['No internet service' 'No' 'Yes' 'Y' 'True']
TechSupport unique values: ['No internet service' 'Yes' 'No' 'Y' 'True']
StreamingTV unique values: ['No internet service' 'No' 'Yes' 'Y' 'True']
StreamingMovies unique values: ['No internet service' 'No' 'Yes' 'Y' 'True']
PaperlessBilling unique values: ['No' 'Yes']


In [ ]:
# Standardize values in binary columns
df.replace(['Y', 'True'], 'Yes', inplace=True)

In [ ]:
# Convert alternative labels to consistent Yes/No format
df.replace('No internet service', 'No', inplace=True)

In [ ]:
# List of binary columns that contain Yes/No values
yes_no_columns = [
    'Partner',
    'Dependents',
    'PhoneService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'PaperlessBilling'
]

# Fill missing values in binary columns (assume missing = No)
for col in yes_no_columns:
    df[col] = df[col].fillna('No')

In [ ]:
# Check the unique values in each binary column after cleaning

for col in yes_no_columns:
    print(col, df[col].unique())

Partner ['No' 'Yes']
Dependents ['Yes' 'No']
PhoneService ['Yes' 'No']
OnlineSecurity ['No' 'Yes']
OnlineBackup ['No' 'Yes']
DeviceProtection ['No' 'Yes']
TechSupport ['No' 'Yes']
StreamingTV ['No' 'Yes']
StreamingMovies ['No' 'Yes']
PaperlessBilling ['No' 'Yes']


In [ ]:
# Convert Yes/No values to numeric format (Yes=1, No=0) for machine learning models

for col in yes_no_columns:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

In [ ]:
# Display the first rows of the converted binary columns

print(df[yes_no_columns].head())


   Partner  Dependents  PhoneService  OnlineSecurity  OnlineBackup  \
0        0           1             1               0             0   
1        1           0             1               0             0   
2        0           0             1               0             1   
3        0           1             1               0             0   
4        1           1             1               1             0   

   DeviceProtection  TechSupport  StreamingTV  StreamingMovies  \
0                 0            0            0                0   
1                 0            1            0                0   
2                 0            0            1                1   
3                 1            0            0                0   
4                 1            0            0                0   

   PaperlessBilling  
0                 0  
1                 1  
2                 0  
3                 0  
4                 1  


In [ ]:
# Verify that the data type of these columns is numeric

print(df[yes_no_columns].dtypes)

Partner             int64
Dependents          int64
PhoneService        int64
OnlineSecurity      int64
OnlineBackup        int64
DeviceProtection    int64
TechSupport         int64
StreamingTV         int64
StreamingMovies     int64
PaperlessBilling    int64
dtype: object




*   ### Converting Churn to Numeric Format




In [ ]:
# Check the distribution and data type of the target variable (Churn)

df['Churn'].value_counts()
df['Churn'].info()

<class 'pandas.core.series.Series'>
Index: 69946 entries, 0 to 69999
Series name: Churn
Non-Null Count  Dtype
--------------  -----
69946 non-null  int64
dtypes: int64(1)
memory usage: 1.1 MB


In [ ]:
# Convert the target variable from categorical values (Yes/No) to numeric format
# Yes -> 1 (customer churned)
# No  -> 0 (customer stayed)

df['Churn'] = df['Churn'].replace({'Yes': 1, 'No': 0})

In [ ]:
# Verify that the conversion was successful

df['Churn'].value_counts()
df['Churn'].info()

<class 'pandas.core.series.Series'>
Index: 69946 entries, 0 to 69999
Series name: Churn
Non-Null Count  Dtype
--------------  -----
69946 non-null  int64
dtypes: int64(1)
memory usage: 1.1 MB


In [ ]:
# Check if there are any remaining missing values in binary columns

print(df[yes_no_columns].isnull().sum())

Partner             0
Dependents          0
PhoneService        0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
PaperlessBilling    0
dtype: int64


In [ ]:
# Identify remaining categorical columns that still need encoding

df.select_dtypes(include='object').columns

Index(['customerID', 'gender', 'MultipleLines', 'InternetService', 'Contract',
       'PaymentMethod'],
      dtype='object')

In [ ]:
# Display all categorical columns for further preprocessing

for col in df.select_dtypes(include='object').columns:
    print("Column:", col)
    print(df[col].unique())
    print("-"*50)

Column: customerID
['CUST00001' 'CUST00002' 'CUST00003' ... 'CUST69998' 'CUST69999'
 'CUST70000']
--------------------------------------------------
Column: gender
['Male' 'Female']
--------------------------------------------------
Column: MultipleLines
['Yes' 'No' 'No phone service']
--------------------------------------------------
Column: InternetService
['No' 'DSL' 'Fiber optic']
--------------------------------------------------
Column: Contract
['Month-to-month' 'One year' 'Two year' 'M-M']
--------------------------------------------------
Column: PaymentMethod
['Mailed check' 'Bank transfer (automatic)' 'Electronic check'
 'Credit card (automatic)']
--------------------------------------------------


*   ###  Gender Feature Preprocessing

In [ ]:
# Check the unique values in the gender column

df['gender'].unique()

array(['Male', 'Female'], dtype=object)

In [ ]:
# Standardize gender values (remove spaces and unify formats)

df['gender'] = df['gender'].str.strip().str.capitalize()
df['gender'] = df['gender'].replace({'F': 'Female', 'M': 'Male'})
df['gender'] = df['gender'].replace({ 'Man': 'Male'})

In [ ]:
# Replace string 'nan' with actual missing values and fill them using the most frequent value

df['gender'] = df['gender'].replace('nan', None)
df['gender'] = df['gender'].fillna(df['gender'].mode()[0])

In [ ]:
# Verify that there are no missing values remaining

print(df['gender'].isnull().sum())

0


In [ ]:
# Check the final unique values after cleaning

print(df['gender'].unique())

['Male' 'Female']


In [ ]:
# Encode gender column to numeric values for machine learning
# Male = 1, Female = 0

df['gender'] = df['gender'].map({'Male':1, 'Female':0})

In [ ]:
# Final check for unique values and missing values

print("Unique values:", df['gender'].unique())
print("Missing values:", df['gender'].isnull().sum())
print("Data type:", df['gender'].dtype)

Unique values: [1 0]
Missing values: 0
Data type: int64


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 69946 entries, 0 to 69999
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        69946 non-null  object 
 1   gender            69946 non-null  int64  
 2   SeniorCitizen     69946 non-null  int64  
 3   Partner           69946 non-null  int64  
 4   Dependents        69946 non-null  int64  
 5   tenure            69946 non-null  float64
 6   PhoneService      69946 non-null  int64  
 7   MultipleLines     69946 non-null  object 
 8   InternetService   69946 non-null  object 
 9   OnlineSecurity    69946 non-null  int64  
 10  OnlineBackup      69946 non-null  int64  
 11  DeviceProtection  69946 non-null  int64  
 12  TechSupport       69946 non-null  int64  
 13  StreamingTV       69946 non-null  int64  
 14  StreamingMovies   69946 non-null  int64  
 15  Contract          69946 non-null  object 
 16  PaperlessBilling  69946 non-null  int64  
 17


*   ### Fix Inconsistent Service Data






In [ ]:
# Check distribution of values in MultipleLines column

df["MultipleLines"].value_counts()

,count
MultipleLines,
Yes,34660
No,27033
No phone service,8253


In [ ]:
# Identify inconsistent rows:
# Customers who have PhoneService = 1 but MultipleLines = "No phone service"
# which is logically incorrect

df[(df["PhoneService"] == 1) &
   (df["MultipleLines"] == "No phone service")].shape

(1320, 21)

In [ ]:
# Create a mask to locate inconsistent records

mask = (df['PhoneService'] == 1) & (df['MultipleLines'] == 'No phone service')

# Fix the inconsistency by replacing "No phone service" with "No"
df.loc[mask, 'MultipleLines'] = 'No'

# Print how many rows were corrected

print(f'Fixed {mask.sum()} inconsistent rows')

Fixed 1320 inconsistent rows


In [ ]:
# Verify the correction by checking value distribution again
print(df['MultipleLines'].value_counts())

MultipleLines
Yes                 34660
No                  28353
No phone service     6933
Name: count, dtype: int64


In [ ]:
# Display dataset structure and data types after cleaning
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 69946 entries, 0 to 69999
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        69946 non-null  object 
 1   gender            69946 non-null  int64  
 2   SeniorCitizen     69946 non-null  int64  
 3   Partner           69946 non-null  int64  
 4   Dependents        69946 non-null  int64  
 5   tenure            69946 non-null  float64
 6   PhoneService      69946 non-null  int64  
 7   MultipleLines     69946 non-null  object 
 8   InternetService   69946 non-null  object 
 9   OnlineSecurity    69946 non-null  int64  
 10  OnlineBackup      69946 non-null  int64  
 11  DeviceProtection  69946 non-null  int64  
 12  TechSupport       69946 non-null  int64  
 13  StreamingTV       69946 non-null  int64  
 14  StreamingMovies   69946 non-null  int64  
 15  Contract          69946 non-null  object 
 16  PaperlessBilling  69946 non-null  int64  
 17

In [ ]:
# Check unique values and distribution in InternetService column

df["InternetService"].value_counts()

,count
InternetService,
Fiber optic,38470
DSL,17579
No,13897


In [ ]:
# Recheck MultipleLines distribution after previous fixes

df["MultipleLines"].value_counts()

,count
MultipleLines,
Yes,34660
No,28353
No phone service,6933


# Standardizing Contract Values

In [ ]:
# Examine the Contract column values to detect inconsistent naming
df["Contract"].value_counts()

,count
Contract,
Month-to-month,41814
One year,14005
Two year,13961
M-M,166


In [ ]:
# Standardize abbreviated contract name to full format

df["Contract"] = df["Contract"].replace({
    "M-M": "Month-to-month"
})

# Verify that the replacement was successful

print(df["Contract"].value_counts())

Contract
Month-to-month    41980
One year          14005
Two year          13961
Name: count, dtype: int64


# **Feature Engineering**
Creating new variables to improve model performance

In [ ]:
# Create a new binary feature indicating whether the customer has internet service
# If InternetService is NOT "No" → value = 1
# If InternetService is "No" → value = 0

df["HasInternet"] = (df["InternetService"] != "No").astype(int)


In [ ]:
# Check the distribution of the newly created HasInternet feature

df["HasInternet"].value_counts()

,count
HasInternet,
1,56049
0,13897


In [ ]:
# Display first 10 rows to verify that HasInternet was created correctly

display(df[['InternetService', 'HasInternet']].head(10))

,InternetService,HasInternet
0,No,0
1,DSL,1
2,DSL,1
3,Fiber optic,1
4,Fiber optic,1
5,Fiber optic,1
6,No,0
7,Fiber optic,1
8,DSL,1
9,No,0



# **Remove Outliers**

*   ### tenure: impossible values (-10, 999 months)
*   ### MonthlyCharges: extreme values up to $1,478




In [ ]:
# Print dataset size before filtering
print(f"Rows before: {len(df):,}")
# Remove unrealistic tenure values (keep only values between 0 and 100 months)
df = df[(df["tenure"] >= 0) & (df["tenure"] <= 100)]
# Remove extreme MonthlyCharges values to keep reasonable pricing range
df = df[df["MonthlyCharges"] <= 200]
# Print dataset size after filtering to see how many rows were removed
print(f"Rows after: {len(df):,}")

Rows before: 69,946
Rows after: 67,376


#  Encoding Categorical Features(One-Hot Encoding)


In [ ]:
# Check the distribution of different payment methods
df["PaymentMethod"].value_counts()

,count
PaymentMethod,
Electronic check,30263
Bank transfer (automatic),13653
Mailed check,13467
Credit card (automatic),9993


In [ ]:
# Apply One-Hot Encoding to PaymentMethod
# This will create a separate binary column for each payment method

df = pd.get_dummies(df, columns=["PaymentMethod"])

In [ ]:
# Verify that the new encoded columns were created successfully
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 67376 entries, 0 to 69999
Data columns (total 25 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   customerID                               67376 non-null  object 
 1   gender                                   67376 non-null  int64  
 2   SeniorCitizen                            67376 non-null  int64  
 3   Partner                                  67376 non-null  int64  
 4   Dependents                               67376 non-null  int64  
 5   tenure                                   67376 non-null  float64
 6   PhoneService                             67376 non-null  int64  
 7   MultipleLines                            67376 non-null  object 
 8   InternetService                          67376 non-null  object 
 9   OnlineSecurity                           67376 non-null  int64  
 10  OnlineBackup                             67376 non-

# Converting Categorical Features to Numeric Values


In [ ]:

# MultipleLines - has order: no service < no < yes
df['MultipleLines'] = df['MultipleLines'].replace({
    'No phone service': 0, 'No': 1, 'Yes': 2
})

# InternetService - has order: no < DSL < Fiber
df['InternetService'] = df['InternetService'].replace({
    'No': 0, 'DSL': 1, 'Fiber optic': 2
})

# Contract - has order: monthly < yearly < two years
df['Contract'] = df['Contract'].replace({
    'Month-to-month': 0, 'One year': 1, 'Two year': 2
})


# Check if there are still any columns with object datatype
print("Remaining object columns:", df.select_dtypes(include='object').columns.tolist())
# Print dataset shape (rows and columns)
print("Shape:", df.shape)


Remaining object columns: ['customerID', 'MultipleLines', 'InternetService', 'Contract']
Shape: (67376, 25)


##  Final Dataset Validation

In [ ]:
# Verify that the encoded columns contain only numeric values
print("MultipleLines:", df['MultipleLines'].unique())
print("InternetService:", df['InternetService'].unique())
print("Contract:", df['Contract'].unique())

MultipleLines: [2 1 0]
InternetService: [0 1 2]
Contract: [0 1 2]



# Convert Encoded Columns to Integer


In [ ]:
# Convert encoded categorical columns to integer type
# This ensures the model treats them as numeric features

df['MultipleLines'] = df['MultipleLines'].astype(int)
df['InternetService'] = df['InternetService'].astype(int)
df['Contract'] = df['Contract'].astype(int)

# Check if there are any remaining object (text) columns
print("Remaining object columns:", df.select_dtypes(include='object').columns.tolist())
# Verify the data types of the converted columns
print(df[['MultipleLines', 'InternetService', 'Contract']].dtypes)

Remaining object columns: ['customerID']
MultipleLines      int64
InternetService    int64
Contract           int64
dtype: object


#**Dataset Final Checks**

In [ ]:

# This helps verify if rows were removed during cleaning
print("=== Shape ===")
print(df.shape)

# Count total missing values in the dataset
print("\n=== Missing Values ===")
print(df.isna().sum().sum(), "total missing values")

# Display the distribution of the target variable (Churn)
# normalize=True shows percentages instead of counts
print("\n=== Churn Distribution ===")
print(df['Churn'].value_counts(normalize=True).round(3))

# This confirms all features are numeric and ready for modeling
print("\n=== Data Types ===")
print(df.dtypes)

=== Shape ===
(67376, 25)

=== Missing Values ===
0 total missing values

=== Churn Distribution ===
Churn
0    0.531
1    0.469
Name: proportion, dtype: float64

=== Data Types ===
customerID                                  object
gender                                       int64
SeniorCitizen                                int64
Partner                                      int64
Dependents                                   int64
tenure                                     float64
PhoneService                                 int64
MultipleLines                                int64
InternetService                              int64
OnlineSecurity                               int64
OnlineBackup                                 int64
DeviceProtection                             int64
TechSupport                                  int64
StreamingTV                                  int64
StreamingMovies                              int64
Contract                                     int64
Pa

# **Feature Engineering**
Create New Useful Feature


In [ ]:


# 1. Create a feature to identify new customers
#  Is the customer new? (less than 6 months)
df['IsNewCustomer'] = (df['tenure'] <= 6).astype(int)

# 2. Create a feature to identify long-term customers
#  Customers with tenure >= 48 months are considered loyal / long-term
df['IsLongTermCustomer'] = (df['tenure'] >= 48).astype(int)

# 3. Average monthly charge based on total charges
# This helps normalize TotalCharges based on how long the customer stayed
df['AvgMonthlyCharge'] = df['TotalCharges'] / (df['tenure'] + 1)

# 4. Number of services subscribed
# This may indicate customer engagement level
service_cols = [
    'PhoneService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
df['TotalServices'] = df[service_cols].sum(axis=1)

# Preview the new features
print(df[['IsNewCustomer', 'IsLongTermCustomer',
          'AvgMonthlyCharge', 'TotalServices']].head(10))

    IsNewCustomer  IsLongTermCustomer  AvgMonthlyCharge  TotalServices
0               1                   0         51.457500              1
1               1                   0         15.433333              2
2               0                   0         41.638605              4
3               0                   0         73.209756              2
4               0                   0         21.136667              3
5               0                   0         35.270526              2
6               0                   0         64.875000              1
7               0                   0         74.055882              1
8               1                   0         55.594286              2
10              0                   0         46.582105              1


In [ ]:
# Save the cleaned dataset after feature engineering
df.to_csv("telco_customer_data_cleaned.csv", index=False)
print(" dataset cleaned !")

 dataset cleaned !


##Handle Negative Total Charges

In [ ]:
# Check if there are negative values in TotalCharges
print("Before ")
print(f"Negtive Numbers  {(df['TotalCharges'] < 0).sum()}")
# Display examples of problematic rows
print(df[df['TotalCharges'] < 0][['tenure', 'MonthlyCharges', 'TotalCharges']].head())

# Replace negative values using the formula: TotalCharges ≈ tenure * MonthlyCharges
negative_mask = df['TotalCharges'] < 0
df.loc[negative_mask, 'TotalCharges'] = (
    df.loc[negative_mask, 'tenure'] *
    df.loc[negative_mask, 'MonthlyCharges']
)

# Verify fix
print("After Fix ")
print(f" Negtive Numbers : {(df['TotalCharges'] < 0).sum()}")
print(f"Min TotalCharges: {df['TotalCharges'].min()}")


Before 
Negtive Numbers  195
      tenure  MonthlyCharges  TotalCharges
137     32.0           56.57        -64.03
237     14.0           48.92        -80.77
701      3.0          115.74        -22.32
749     12.0           44.17        -98.28
1250     6.0           27.50        -39.63
After Fix 
 Negtive Numbers : 0
Min TotalCharges: 0.0


In [ ]:
# Save the final version of the dataset
df.to_csv("telco_customer_data.csv", index=False)


In [ ]:
# Ensures all features are numeric and ready for ML models
df.dtypes

,0
customerID,object
gender,int64
SeniorCitizen,int64
Partner,int64
Dependents,int64
tenure,float64
PhoneService,int64
MultipleLines,int64
InternetService,int64
OnlineSecurity,int64


In [ ]:
# Check if any missing values remain in the dataset
df.isna().sum().sort_values(ascending=False)

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [ ]:
# It does not help in predicting churn
df = df.drop(columns=["customerID"])

In [ ]:
# Final verification of data types
df.dtypes

,0
gender,int64
SeniorCitizen,int64
Partner,int64
Dependents,int64
tenure,float64
PhoneService,int64
MultipleLines,int64
InternetService,int64
OnlineSecurity,int64
OnlineBackup,int64


###Validate Engineered Features

in notbook 2 we noticed that there are a - values in AvgMonthlyCharge

In [ ]:
# Check if there are any negative values in AvgMonthlyCharge
(df["AvgMonthlyCharge"] < 0).sum()

np.int64(195)

In [ ]:
# Display rows with negative values if they exist
df[df["AvgMonthlyCharge"] < 0]

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Churn,HasInternet,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,IsNewCustomer,IsLongTermCustomer,AvgMonthlyCharge,TotalServices
137,1,0,0,0,32.0,1,2,2,1,0,...,0,1,True,False,False,False,0,0,-1.940303,2
237,1,1,0,0,14.0,1,2,2,0,0,...,0,1,False,True,False,False,0,0,-5.384667,3
701,0,0,0,1,3.0,1,2,1,0,0,...,0,1,True,False,False,False,1,0,-5.580000,1
749,0,0,1,0,12.0,1,2,2,1,1,...,1,1,False,False,True,False,0,0,-7.560000,5
1250,0,1,0,0,6.0,1,1,2,0,1,...,0,1,True,False,False,False,1,0,-5.661429,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68026,1,0,1,0,47.0,1,2,2,0,0,...,0,1,False,False,False,True,0,0,-2.028750,1
68158,0,0,1,0,10.0,1,1,1,0,0,...,0,1,False,False,True,False,0,0,-2.175455,1
69333,0,0,0,0,56.0,1,1,1,0,0,...,1,1,False,True,False,False,0,1,-1.015088,1
69871,0,0,1,0,31.0,1,1,1,0,0,...,1,1,False,True,False,False,0,0,-2.299375,3


In [ ]:
# Summary statistics for TotalCharges
df["TotalCharges"].describe()

,TotalCharges
count,67376.000000
mean,1119.984806
std,1046.313623
min,0.000000
25%,395.835000
50%,820.430000
75%,1497.760000
max,8459.280000


In [ ]:
# Shows distribution of customer lifetime
df["tenure"].describe()

,tenure
count,67376.000000
mean,22.918428
std,15.276531
min,1.000000
25%,10.000000
50%,20.000000
75%,34.000000
max,72.000000


Recalculate AvgMonthlyCharge

In [ ]:
# Recalculate average monthly charge to ensure accuracy
df["AvgMonthlyCharge"] = df["TotalCharges"] / (df["tenure"] + 1)

In [ ]:
# Check distribution of the recalculated feature
df["AvgMonthlyCharge"].describe()

,AvgMonthlyCharge
count,67376.000000
mean,44.863760
std,24.864617
min,0.000000
25%,26.671684
50%,37.477337
75%,55.335121
max,116.351250


In [ ]:
# Verify again that no negative values exist
df[df["AvgMonthlyCharge"] < 0]

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Churn,HasInternet,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,IsNewCustomer,IsLongTermCustomer,AvgMonthlyCharge,TotalServices


rename payment methods

In [ ]:
# Ensures all features are correctly formatted
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 67376 entries, 0 to 69999
Data columns (total 28 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   gender                                   67376 non-null  int64  
 1   SeniorCitizen                            67376 non-null  int64  
 2   Partner                                  67376 non-null  int64  
 3   Dependents                               67376 non-null  int64  
 4   tenure                                   67376 non-null  float64
 5   PhoneService                             67376 non-null  int64  
 6   MultipleLines                            67376 non-null  int64  
 7   InternetService                          67376 non-null  int64  
 8   OnlineSecurity                           67376 non-null  int64  
 9   OnlineBackup                             67376 non-null  int64  
 10  DeviceProtection                         67376 non-

In [ ]:
# Convert boolean values to numeric (1/0) for machine learning compatibility
df["PaymentMethod_Bank transfer (automatic)"] = df["PaymentMethod_Bank transfer (automatic)"].replace({
    True: 1,
    False: 0
})

In [ ]:
# Verify transformation
df["PaymentMethod_Bank transfer (automatic)"].value_counts()

,count
PaymentMethod_Bank transfer (automatic),
0,53723
1,13653


In [ ]:
# Convert Electronic Check payment method column to numeric
df["PaymentMethod_Electronic check"] = df["PaymentMethod_Electronic check"].replace({
    True: 1,
    False: 0
})

In [ ]:
# Check distribution for Electronic Check payment method
df["PaymentMethod_Electronic check"].value_counts()

,count
PaymentMethod_Electronic check,
0,37113
1,30263


In [ ]:
# Convert Credit Card payment method column from boolean to numeric

df["PaymentMethod_Credit card (automatic)"] = df["PaymentMethod_Credit card (automatic)"].replace({
    True: 1,
    False: 0
})

In [ ]:
# Check distribution for Electronic Check payment method
df["PaymentMethod_Credit card (automatic)"].value_counts()

,count
PaymentMethod_Credit card (automatic),
0,57383
1,9993


In [ ]:
# Convert Mailed Check payment method column to numeric
df["PaymentMethod_Mailed check"] = df["PaymentMethod_Mailed check"].replace({
    True: 1,
    False: 0
})

In [ ]:
# Check distribution for Electronic Check payment method
df["PaymentMethod_Mailed check"].value_counts()

,count
PaymentMethod_Mailed check,
0,53909
1,13467


In [ ]:
# Delete the HasInternet column
df = df.drop(columns=['HasInternet'])
print(df.shape)

(67376, 27)


In [ ]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TotalCharges,Churn,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,IsNewCustomer,IsLongTermCustomer,AvgMonthlyCharge,TotalServices
0,1,0,0,1,3.0,1,2,0,0,0,...,205.83,1,0,0,0,1,1,0,51.457500,1
1,1,1,1,0,2.0,1,2,1,0,0,...,46.30,0,1,0,0,0,1,0,15.433333,2
2,0,0,0,0,42.0,1,2,1,0,1,...,1790.46,1,0,0,1,0,0,0,41.638605,4
3,0,0,0,1,40.0,1,2,2,0,0,...,3001.60,0,0,0,1,0,0,0,73.209756,2
4,1,1,1,1,17.0,1,1,2,1,0,...,380.46,1,0,0,1,0,0,0,21.136667,3


**Save Final Cleaned Dataset**

In [ ]:
# Save the final cleaned dataset ready for analysis and modeling
df.to_csv("telco_customer_data_cleaned.csv", index=False)

# **Great work my team!**
**the cleaned dataset uploaded to the github repository:**
